In [ ]:
import os
import cv2
import numpy as np
import ipywidgets as widgets
from IPython.display import display, Video
import matplotlib.pyplot as plt

In [5]:
entry_mask_path = "../src/mask/road_mask/normanniles1_entry_mask.png"
roundpoint_mask_path = "../src/mask/road_mask/normanniles1_roundpoint_mask.png"
exit_mask_path = "../src/mask/road_mask/normanniles1_exit_mask.png"

# video_rel_path = "./data/videos/normanniles1/normanniles1_2025-10-21-14-00-45.mp4" # day, entry : heavy delay, exit : light delay

# video_rel_path = "data/videos/normanniles1/normanniles1_2025-10-20-13-20-45.mp4" # day, entry : light delay, exit : free flowing
# video_rel_path = "data/videos/normanniles1/normanniles1_2025-10-20-15-08-45.mp4" # day, entry : heavy delay, exit : free flowing
video_rel_path = "../data/videos/normanniles1/normanniles1_2025-10-21-10-55-45.mp4" # day, entry : heavy delay, exit : heavy delay
# video_rel_path = "data/videos/normanniles1/normanniles1_2025-10-20-06-01-45.mp4" # night, entry : free flowing, exit : free flowing (completely empty)
# video_rel_path = "data/videos/normanniles1/normanniles1_2025-10-21-10-58-45.mp4" # day, entry : moderate delay, exit : moderate delay
video_path = os.path.realpath(video_rel_path)

In [6]:
def create_background_model(video_path, num_frames=50):
    """
    Generates a background image (empty road) by taking the median of random frames.
    """
    cap = cv2.VideoCapture(video_path)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    random_indices = np.random.choice(total_frames, num_frames, replace=False)
    frames = []
    
    print("Generating background model...")
    for idx in random_indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ret, frame = cap.read()
        if ret:
            frames.append(frame)
    cap.release()
    
    median_frame = np.median(frames, axis=0).astype(dtype=np.uint8)
    return cv2.cvtColor(median_frame, cv2.COLOR_BGR2GRAY)


def calculate_occupancy(frame_gray, background_gray, mask, threshold=25):
    blurred_frame = cv2.GaussianBlur(frame_gray, (5, 5), 0)
    blurred_bg = cv2.GaussianBlur(background_gray, (5, 5), 0)

    diff_intensity = cv2.absdiff(blurred_bg, blurred_frame)
    _, binary_intensity = cv2.threshold(
        diff_intensity, threshold, 255, cv2.THRESH_BINARY)

    edges_frame = cv2.Canny(blurred_frame, 50, 150)
    edges_bg = cv2.Canny(blurred_bg, 50, 150)

    edges_diff = cv2.subtract(edges_frame, edges_bg)

    combined_binary = cv2.bitwise_or(binary_intensity, edges_diff)

    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (5, 5))
    closed_mask = cv2.morphologyEx(combined_binary, cv2.MORPH_CLOSE, kernel)
    dilated_mask = cv2.dilate(closed_mask, kernel, iterations=2)

    motion_in_zone = cv2.bitwise_and(dilated_mask, dilated_mask, mask=mask)

    total_pixels_mask = cv2.countNonZero(mask)
    if total_pixels_mask == 0:
        return 0.0

    occupied_pixels = cv2.countNonZero(motion_in_zone)
    return occupied_pixels / total_pixels_mask

In [7]:
mask_entry = cv2.imread(entry_mask_path, cv2.IMREAD_GRAYSCALE)
mask_round = cv2.imread(roundpoint_mask_path, cv2.IMREAD_GRAYSCALE)
mask_exit = cv2.imread(exit_mask_path, cv2.IMREAD_GRAYSCALE)

mask_total = cv2.bitwise_or(mask_entry, mask_round)
mask_total = cv2.bitwise_or(mask_total, mask_exit)

masks_dict = {
    0: mask_entry,
    1: mask_exit, # Attention à l'ordre demandé : Entrée, Sortie, Rond-point, Total
    2: mask_round,
    3: mask_total
}

background = create_background_model(video_path)

Generating background model...


In [ ]:
cap = cv2.VideoCapture(video_path)
fps = cap.get(cv2.CAP_PROP_FPS)

occupancy_data = np.zeros((60, 4))

print("Starting frame-by-frame analysis...")

for sec in range(60):
    # We will store the values for the 3 frames of this second
    temp_values = np.zeros((3, 4)) 
    
    # We take 3 frames evenly distributed in the second (e.g., t+0, t+0.33, t+0.66)
    offsets = [0.0, 0.33, 0.66]
    
    for i, offset in enumerate(offsets):
        target_frame_idx = int((sec + offset) * fps)
        cap.set(cv2.CAP_PROP_POS_FRAMES, target_frame_idx)
        ret, frame = cap.read()
        
        if not ret:
            break # Premature end of video
            
        frame_gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        
        # Calculation for the 4 masks
        for col_idx in range(4):
            val = calculate_occupancy(frame_gray, background, masks_dict[col_idx])
            temp_values[i, col_idx] = val
            
    # Average of the 3 frames to get the value for the second
    occupancy_data[sec] = np.mean(temp_values, axis=0)

cap.release()
print("Analysis completed.")

In [ ]:
zones_names = ["Entry", "Exit", "Roundabout", "Total Combined"]
print("\n--- Occupancy Averages Over the Minute ---")
for i, name in enumerate(zones_names):
    print(f"{name} : {np.mean(occupancy_data[:, i]):.4f} ({np.mean(occupancy_data[:, i])*100:.1f}%)")

plt.figure(figsize=(12, 12))
time_axis = np.arange(60)

for i, name in enumerate(zones_names):
    plt.subplot(4, 1, i + 1)
    plt.plot(time_axis, occupancy_data[:, i], label=name, alpha=0.8)
    plt.title(f"Occupancy Rate: {name}")
    plt.xlabel("Time (seconds)")
    plt.ylabel("Occupancy Rate")
    plt.ylim(0, 1.0)
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
video_widget = widgets.Output()
with video_widget:
    display(Video(video_path, width=1600))
display(video_widget)

In [ ]:
def process_one_minute_data(minute_data):
    features = []

    # Iterate over the 4 zones (Entry, Exit, Roundpoint, Total)
    for zone_idx in range(4):
        zone_series = minute_data[:, zone_idx]

        # 1. Mean (General occupancy)
        mu = np.mean(zone_series)

        # 2. Standard Deviation (Volatility)
        sigma = np.std(zone_series)

        # 3. Max (Peak congestion)
        maximum = np.max(zone_series)

        # 4. Min (Base load)
        minimum = np.min(zone_series)

        # 5. Slope (Trend: is it filling up or emptying?)
        # We fit a line y = ax + b, we want 'a'
        x = np.arange(len(zone_series))
        if len(zone_series) > 1:
            slope, _ = np.polyfit(x, zone_series, 1)
        else:
            slope = 0.0

        features.extend([mu, sigma, maximum, minimum, slope])

    return np.array(features)

features_vector = process_one_minute_data(occupancy_data)
print(f"Mean : {features_vector[0]:.4f}, Std Dev : {features_vector[1]:.4f}, Max : {features_vector[2]:.4f}, Min : {features_vector[3]:.4f}, Slope : {features_vector[4]:.6f}")